# Task 1:

In [25]:
!pip install datasets==2.19.1

In [26]:
import datasets
dataset = datasets.load_dataset("conll2003")

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [27]:
train_sentances = dataset['train']['tokens']
train_labels = dataset['train']['ner_tags']
test_sentances = dataset['test']['tokens']
test_labels = dataset['test']['ner_tags']
valid_sentances = dataset['validation']['tokens']
valid_labels = dataset['validation']['ner_tags']

In [28]:
dataset['train']['tokens'][0]

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

In [29]:
dataset['train']['ner_tags'][0]

[3, 0, 7, 0, 0, 0, 7, 0, 0]

In [30]:
id_to_name_map = dataset["train"].features["ner_tags"].feature.names

def int_to_str_tags(int_tags):
    return [id_to_name_map[tag_id] for tag_id in int_tags]

train_labels_str = [int_to_str_tags(tags) for tags in dataset['train']['ner_tags']]
valid_labels_str = [int_to_str_tags(tags) for tags in dataset['validation']['ner_tags']]

In [31]:
train_labels[0]

[3, 0, 7, 0, 0, 0, 7, 0, 0]

In [32]:
from collections import Counter
def build_vocabularies(sentences, min_word_freq=1):
    word_counter = Counter()
    tag_counter = Counter()

    for sentence in sentences:
        for word, tag in sentence:
            word_counter[word] += 1
            tag_counter[tag] += 1

    word_to_ix = {'<PAD>': 0, '<UNK>': 1}
    for word, count in word_counter.items():
        if count >= min_word_freq:
            word_to_ix[word] = len(word_to_ix)

    tag_to_ix = {'<PAD>': 0}
    for tag in tag_counter.keys():
        tag_to_ix[tag] = len(tag_to_ix)

    return word_to_ix, tag_to_ix


In [33]:
combined_train_data = []
for i in range(len(train_sentances)):
    combined_train_data.append(list(zip(train_sentances[i], train_labels_str[i])))

combined_valid_data = []
for i in range(len(valid_sentances)):
    combined_valid_data.append(list(zip(valid_sentances[i], valid_labels_str[i])))

word_to_ix, tag_to_ix = build_vocabularies(combined_train_data)

In [34]:
print(len(word_to_ix))
print(len(tag_to_ix))

23625
10


# Task 2:

In [35]:
import torch

class NERDataset(torch.utils.data.Dataset):
  def __init__(self, tokens, labels, word_to_ix, tag_to_ix):
      self.tokens = tokens
      self.labels = labels
      self.word_to_ix = word_to_ix
      self.tag_to_ix = tag_to_ix

  def __len__(self):
      return len(self.tokens)

  def __getitem__(self, idx):
      sentence = self.tokens[idx]

      word_indices = [self.word_to_ix.get(word, self.word_to_ix['<UNK>']) for word, _ in sentence]
      tag_indices = [self.tag_to_ix[label] for _, label in sentence]

      return torch.tensor(word_indices), torch.tensor(tag_indices)


In [36]:
train_dataset = NERDataset(combined_train_data, combined_train_data, word_to_ix, tag_to_ix)
valid_dataset = NERDataset(combined_valid_data, combined_valid_data, word_to_ix, tag_to_ix)

In [37]:
def collate_fn(batch):
    sentences = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    padded_sentences = torch.nn.utils.rnn.pad_sequence(sentences, batch_first=True, padding_value=word_to_ix['<PAD>'])

    padded_labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=tag_to_ix['<PAD>'])

    return padded_sentences, padded_labels

In [38]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

for sentences, labels in train_dataloader:
    print("Sentences shape:", sentences.shape)
    print("Labels shape:", labels.shape)
    break

Sentences shape: torch.Size([32, 44])
Labels shape: torch.Size([32, 44])


# Task 3: Xây dựng Mô hình RNN

In [39]:
import torch.nn as nn

class NER_RNN_Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_size):
        super(NER_RNN_Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word_to_ix['<PAD>'])
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, output_size)

    def forward(self, sentence):
        embeds = self.embedding(sentence)

        rnn_out, _ = self.rnn(embeds)

        tag_space = self.hidden2tag(rnn_out.reshape(-1, rnn_out.shape[2]))

        tag_scores = tag_space.view(sentence.shape[0], sentence.shape[1], -1)

        return tag_scores

In [40]:
vocab_size = len(word_to_ix)
embedding_dim = 100
hidden_dim = 128
output_size = len(tag_to_ix)

model = NER_RNN_Model(vocab_size, embedding_dim, hidden_dim, output_size)

print(model)

NER_RNN_Model(
  (embedding): Embedding(23625, 100, padding_idx=0)
  (rnn): RNN(100, 128, batch_first=True)
  (hidden2tag): Linear(in_features=128, out_features=10, bias=True)
)


# Task 4: Huấn luyện Mô hình

In [41]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=tag_to_ix['<PAD>'])

In [42]:
num_epochs = 7

print("Bắt đầu huấn luyện")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for sentences, labels in train_dataloader:
        # 1. Xóa gradient cũ
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(sentences)

        # Reshape outputs và labels cho CrossEntropyLoss
        outputs = outputs.view(-1, output_size) # Chuyển thành tổng số token trong batch, output_size
        labels = labels.view(-1)             # Chuyển thành tổng số token trong batch

        # 3. Tính loss
        loss = loss_fn(outputs, labels)

        # 4. Backward pass
        loss.backward()

        # 5. Cập nhật trọng số
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("Huấn luyện hoàn tất.")

Bắt đầu huấn luyện
Epoch [1/7], Loss: 0.6325
Epoch [2/7], Loss: 0.3690
Epoch [3/7], Loss: 0.2543
Epoch [4/7], Loss: 0.1835
Epoch [5/7], Loss: 0.1351
Epoch [6/7], Loss: 0.1005
Epoch [7/7], Loss: 0.0749
Huấn luyện hoàn tất.


# Task 5: Đánh giá Mô hình

In [43]:
def evaluate(model, dataloader, loss_fn, tag_to_ix, output_size):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    pad_tag_idx = tag_to_ix['<PAD>']

    with torch.no_grad():
        for sentences, labels in dataloader:
            outputs = model(sentences)

            # Reshape outputs và labels cho CrossEntropyLoss
            outputs_flat = outputs.view(-1, output_size)
            labels_flat = labels.view(-1)

            loss = loss_fn(outputs_flat, labels_flat)
            total_loss += loss.item()

            # Lấy dự đoán bằng cách áp dụng argmax trên chiều cuối cùng
            predictions = torch.argmax(outputs, dim=2)

            # So sánh dự đoán với nhãn thật
            non_pad_mask = (labels != pad_tag_idx)
            correct_predictions = (predictions == labels) & non_pad_mask

            total_correct += correct_predictions.sum().item()
            total_tokens += non_pad_mask.sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = total_correct / total_tokens if total_tokens > 0 else 0
    return avg_loss, accuracy

In [44]:
val_loss, val_accuracy = evaluate(model, valid_dataloader, loss_fn, tag_to_ix, output_size)
print(f"Validation Loss: {val_loss:.4f} \nValidation Accuracy: {val_accuracy:.4f}")

Validation Loss: 0.2676 
Validation Accuracy: 0.9218


In [45]:
def predict_sentence(sentence_tokens, model, word_to_ix, id_to_name_map, pad_token='<PAD>', unk_token='<UNK>'):
    model.eval()

    # Chuyển đổi từ thành chỉ số
    word_indices = [word_to_ix.get(word, word_to_ix[unk_token]) for word in sentence_tokens]

    # Chuyển thành tensor và thêm batch dimension
    input_tensor = torch.tensor(word_indices, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_tensor)

    # Lấy dự đoán và chuyển đổi thành chỉ số nhãn
    predictions = torch.argmax(outputs, dim=2).squeeze(0) # Xóa batch dimension

    # Chuyển đổi chỉ số nhãn thành tên nhãn chuỗi
    ix_to_tag = {v: k for k, v in tag_to_ix.items()}

    predicted_tags = [ix_to_tag[idx.item()] for idx in predictions]

    # Kết hợp từ gốc với nhãn dự đoán
    result = list(zip(sentence_tokens, predicted_tags))
    return result

In [46]:
# Ví dụ sử dụng predict_sentence
sample_sentence = dataset['test']['tokens'][0]
predicted_ner_tags = predict_sentence(sample_sentence, model, word_to_ix, id_to_name_map)
print(f"Sample Sentence: {sample_sentence}")
print(f"Predicted NER Tags: {predicted_ner_tags}")

sample_sentence_2 = ['Google', 'was', 'founded', 'by', 'Larry', 'Page', 'and', 'Sergey', 'Brin', 'in', 'California', '.']
predicted_ner_tags_2 = predict_sentence(sample_sentence_2, model, word_to_ix, id_to_name_map)
print(f"\nSample Sentence 2: {sample_sentence_2}")
print(f"Predicted NER Tags 2: {predicted_ner_tags_2}")

Sample Sentence: ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']
Predicted NER Tags: [('SOCCER', 'O'), ('-', 'O'), ('JAPAN', 'B-ORG'), ('GET', 'I-ORG'), ('LUCKY', 'I-ORG'), ('WIN', 'O'), (',', 'O'), ('CHINA', 'O'), ('IN', 'O'), ('SURPRISE', 'B-ORG'), ('DEFEAT', 'I-ORG'), ('.', 'O')]

Sample Sentence 2: ['Google', 'was', 'founded', 'by', 'Larry', 'Page', 'and', 'Sergey', 'Brin', 'in', 'California', '.']
Predicted NER Tags 2: [('Google', 'B-ORG'), ('was', 'O'), ('founded', 'O'), ('by', 'O'), ('Larry', 'B-PER'), ('Page', 'I-PER'), ('and', 'O'), ('Sergey', 'O'), ('Brin', 'I-PER'), ('in', 'O'), ('California', 'B-LOC'), ('.', 'O')]


In [47]:
!pip install seqeval

In [48]:
from seqeval.metrics import classification_report

ix_to_tag = {v: k for k, v in tag_to_ix.items()}

all_true_labels = []
all_pred_labels = []

pad_tag_idx = tag_to_ix['<PAD>']

model.eval()
with torch.no_grad():
    for sentences, labels in valid_dataloader:
        outputs = model(sentences)
        predictions = torch.argmax(outputs, dim=2)

        for i in range(len(sentences)):
            true_tags = []
            pred_tags = []
            for j in range(len(labels[i])):
                if labels[i][j].item() != pad_tag_idx:
                    true_tags.append(ix_to_tag[labels[i][j].item()])
                    pred_tags.append(ix_to_tag[predictions[i][j].item()])
            all_true_labels.append(true_tags)
            all_pred_labels.append(pred_tags)
print(classification_report(all_true_labels, all_pred_labels))

              precision    recall  f1-score   support

         LOC       0.75      0.76      0.75      1837
        MISC       0.58      0.64      0.61       922
         ORG       0.35      0.71      0.47      1341
         PER       0.69      0.69      0.69      1842

   micro avg       0.57      0.71      0.63      5942
   macro avg       0.59      0.70      0.63      5942
weighted avg       0.62      0.71      0.65      5942

